# Notion Data Extractor

이 노트북은 Notion API를 사용하여 특정 데이터베이스나 페이지의 내용을 가져오고 로컬 파일로 저장합니다.

### 준비 사항
1. [Notion My Integrations](https://www.notion.so/my-integrations)에서 새로운 인테그레이션을 생성하고 **Internal Integration Token**을 준비하세요.
2. 가져오려는 페이지나 데이터베이스를 해당 인테그레이션에 **Connections(연결)**를 통해 공유해야 합니다.

In [ ]:
from notion_client import Client
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

NOTION_TOKEN = os.getenv("NOTION_TOKEN")
notion = Client(auth=NOTION_TOKEN)

output_dir = Path("extracted_notion_data")
output_dir.mkdir(exist_ok=True)

## 1. 데이터베이스 쿼리
데이터베이스 ID를 사용하여 해당 데이터베이스의 모든 항목을 가져옵니다.

In [ ]:
def query_notion_database(database_id):
    """데이터베이스의 모든 페이지를 가져옵니다."""
    results = []
    has_more = True
    start_cursor = None
    
    while has_more:
        response = notion.databases.query(
            database_id=database_id,
            start_cursor=start_cursor
        )
        results.extend(response["results"])
        has_more = response["has_more"]
        start_cursor = response["next_cursor"]
        
    return results

# 사용법 예시
db_id = "input_your_database_id_here"
if db_id != "input_your_database_id_here":
    db_results = query_notion_database(db_id)
    with open(output_dir / "database_raw.json", "w", encoding="utf-8") as f:
        json.dump(db_results, f, ensure_ascii=False, indent=2)
    print(f"데이터베이스 {len(db_results)}개의 항목을 가져와 저장했습니다.")

## 2. 페이지 내용(Blocks) 추출
페이지 ID를 사용하여 페이지 내의 모든 블록(텍스트, 이미지 등)을 가져옵니다.

In [ ]:
def get_page_blocks(page_id):
    """페이지 내의 모든 블록을 가져옵니다."""
    blocks = []
    has_more = True
    start_cursor = None
    
    while has_more:
        response = notion.blocks.children.list(
            block_id=page_id,
            start_cursor=start_cursor
        )
        blocks.extend(response["results"])
        has_more = response["has_more"]
        start_cursor = response["next_cursor"]
        
    return blocks

def blocks_to_text(blocks):
    """블록 리스트에서 텍스트만 추출하여 간단한 마크다운 형식으로 변환합니다."""
    text_content = ""
    for block in blocks:
        block_type = block["type"]
        if block_type in ["paragraph", "heading_1", "heading_2", "heading_3", "bulleted_list_item", "numbered_list_item"]:
            content = block[block_type].get("rich_text", [])
            if content:
                plain_text = "".join([t["plain_text"] for t in content])
                if block_type == "heading_1": text_content += f"# {plain_text}\n\n"
                elif block_type == "heading_2": text_content += f"## {plain_text}\n\n"
                elif block_type == "heading_3": text_content += f"### {plain_text}\n\n"
                else: text_content += f"{plain_text}\n\n"
    return text_content

# 사용법 예시
target_page_id = "input_your_page_id_here"
if target_page_id != "input_your_page_id_here":
    blocks = get_page_blocks(target_page_id)
    markdown_text = blocks_to_text(blocks)
    
    file_path = output_dir / f"{target_page_id}.md"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(markdown_text)
    print(f"페이지 내용을 마크다운으로 저장했습니다: {file_path}")